# Lesson 1 - Framing the Problem, Metrics, and Honest Baselines

Machine learning begins not with algorithms, but with framing.  
Every project should start by articulating the **prediction task** in precise terms: what one row of data represents, what the target variable is, and how errors will be evaluated. This stage is often neglected in practice, but it establishes the ground rules for the entire modeling process.

### 1.1 Defining the Task

In supervised learning, we assume that measurable features $X$ contain information about a target variable $y$.  
When the target is continuous (for example, the sale price of a house), the task is called **regression**.

A well-framed problem statement should answer at least the following questions:

- **Unit of analysis.** What does each row represent? (e.g., *one property sold*).
- **Target definition.** Exactly what are we predicting, including units? (e.g., *sale price in US dollars at the time of sale*).
- **Availability.** Is the target observable at training time, and would it be available at prediction time?  
  Beware *data leakage*, where you accidentally use future information that would not be known at inference.

### 1.2 Measuring Error

A model's usefulness depends on how its errors align with stakeholder needs. Different error metrics encode different views of what "good performance" means.

- **Mean Absolute Error (MAE).**  
  Defined as  
  $$
  \text{MAE} = \frac{1}{n}\sum_{i=1}^n |y_i - \hat{y}_i|
  $$  
  Interpreted as "the average size of a typical miss." Robust to outliers and easy to explain.

- **Root Mean Squared Error (RMSE).**  
  Defined as  
  $$
  \text{RMSE} = \sqrt{\frac{1}{n}\sum_{i=1}^n (y_i - \hat{y}_i)^2}
  $$  
  Penalizes large errors more heavily. Appropriate when occasional large deviations are especially costly.

- **Coefficient of Determination (R²).**  
  Defined as  
  $$
  R^2 = 1 - \frac{\sum (y_i - \hat{y}_i)^2}{\sum (y_i - \bar{y})^2}
  $$  
  Measures the proportion of variance explained relative to a baseline that always predicts the mean. Intuitive, but less reliable when the target distribution shifts.

> **Guideline:** Choose one *primary* metric that reflects the true cost of error, and track at least one *secondary* metric for robustness.

### 1.3 Baselines

Before deploying complex models, it is essential to compute **baselines**.  
A baseline is a deliberately simple predictor, often the mean or median of the target in the training set.

- Purpose: to establish a "floor" of expected performance.  
- Interpretation: if your sophisticated model performs no better than the median predictor, either there is no signal in the features or your modeling pipeline is flawed.

### 1.4 Guardrails for Honest Evaluation

To maintain rigor:

1. **Split early.** Divide data into training and test sets before performing any preprocessing or modeling.  
2. **Use pipelines.** Encapsulate preprocessing and modeling in a single object, ensuring consistent transformations at training and inference.  
3. **Ensure reproducibility.** Fix random seeds, record software versions, and, when possible, snapshot the dataset.

> **Mantra:** *Measure what you do. Do not rely on intuition alone; enforce evidence through metrics and baselines.*

In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

data = fetch_california_housing(as_frame=True)
X = data.frame.drop(columns=["MedHouseVal"])
y = data.frame["MedHouseVal"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=7)

baseline = DummyRegressor(strategy="median").fit(X_train, y_train)
pred = baseline.predict(X_test)

rmse = mean_squared_error(y_test, pred) ** 0.5
mae = mean_absolute_error(y_test, pred)
r2 = r2_score(y_test, pred)

print(f"RMSE: {round(rmse, 3)}, MAE: {round(mae, 3)}, R2: {round(r2, 3)}")

RMSE: 1.203, MAE: 0.897, R2: -0.076


## Pre-Lesson 2 - Dataset Mimicry: Crafting Synthetic Features, Noise, and Missingness

Data scientists often need “toy universes”: simplified or mimicked datasets where principles can be tested 
without leaking insights from the real problem. Constructing such datasets develops a deeper intuition for 
what distributions look like and how different types of corruption affect modeling.

### Why mimic data?

1. **Teaching and communication.** A simple, controlled dataset illustrates principles better than a messy real one.  
2. **Pipeline testing.** Synthetic columns let you check if imputation, encoding, or scaling behave as intended.  
3. **Resilience training.** By injecting missing values or noise, you prepare your models for the messiness of real-world data.  
4. **Conceptual clarity.** Building data forces you to think about distributions, cardinalities, and interactions explicitly.

### 2.0 Tools for mimicry

We rely mainly on:
- `numpy.random`: generates numbers from common distributions (uniform, normal, binomial).  
- `pandas`: integrates random values into tabular structures.  
- Controlled corruption: replacing values with `np.nan` or remapping categories.

### 2.1 Generating synthetic numeric features

- Uniform distribution: flat spread across an interval.  
- Normal distribution: bell-shaped with mean and standard deviation.  
- Skewed distributions: exponential, lognormal.  


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

In [ ]:
# Set sample size for synthetic dataset generation
n = 1000

# Initialize random number generator with seed for reproducibility in simulations
rng = np.random.default_rng(42)

# Create a pandas DataFrame to mimic real data with synthetic features
mimic = pd.DataFrame(
    data={
        # Generate discrete uniform distribution for age (integer values between 18 and 79)
        "age": rng.integers(low=18, high=80, size=n),  # discrete uniform
        # Generate normal distribution for income (mean 50k, std 15k, continuous values)
        "income": rng.normal(loc=50_000, scale=15_000, size=n),  # normal
        # Generate lognormal distribution for house size (skewed positive, mean log 7.5, sigma 0.5)
        "house_size": rng.lognormal(mean=7.5, sigma=0.5, size=n),  # skewed
    }
)

mimic.head()

,age,income,house_size
0,23,71498.217599,2396.240996
1,65,51372.802673,1914.654023
2,58,58711.656430,2132.332584
3,45,49148.252084,2667.254020
4,44,47443.886283,1206.232133


### 2.2 Crafting categorical features

- Categories can be drawn from lists with given probabilities, allowing you to simulate low- and high-cardinality scenarios.

In [ ]:
# Define a list of city names for categorical feature simulation
cities = ["Castle Rock", "Maine", "Belo Horizonte", "London"]

# Sample from the cities list with specified probabilities to create a skewed categorical distribution
mimic["city"] = rng.choice(cities, size=n, p=[0.4, 0.3, 0.2, 0.1])

# Generate a list of neighborhood names to simulate high-cardinality categorical feature
neighbourhoods = [f"NBH_{i}" for i in np.arange(1, 51)]  # High-cardinality example

# Randomly assign neighborhoods to each row, demonstrating uniform sampling for high-cardinality
mimic["neighbourhoods"] = rng.choice(neighbourhoods, size=n)
mimic.head()

,age,income,house_size,city,neighbourhoods
0,23,71498.217599,2396.240996,Maine,NBH_15
1,65,51372.802673,1914.654023,Belo Horizonte,NBH_40
2,58,58711.656430,2132.332584,Castle Rock,NBH_31
3,45,49148.252084,2667.254020,Castle Rock,NBH_12
4,44,47443.886283,1206.232133,Maine,NBH_36


### 2.3 Injecting missingness

- You can deliberately set some values to NaN, either uniformly at random or according to a condition (to simulate MNAR).

In [ ]:
# Generate a boolean mask for MCAR missingness: randomly select 10% of rows for 'income' column
mask = rng.random(n) < 0.1
# Set 'income' to NaN where mask is True, simulating missing completely at random (MCAR)
mimic.loc[mask, "income"] = np.nan

# Create a conditional mask for MNAR missingness: higher probability for 'house_size' if age > 60
mask2 = (mimic["age"] > 60) & (rng.random(n) < 0.2)
# Set 'house_size' to NaN based on the conditional mask, simulating missing not at random (MNAR)
mimic.loc[mask2, "house_size"] = np.nan

mimic.isna().sum()

age                0
income            88
house_size        71
city               0
neighbourhoods     0
dtype: int64

In [ ]:
display(mimic)

,age,income,house_size,city,neighbourhoods
0,23,71498.217599,2396.240996,Maine,NBH_15
1,65,51372.802673,1914.654023,Belo Horizonte,NBH_40
2,58,58711.656430,2132.332584,Castle Rock,NBH_31
3,45,49148.252084,2667.254020,Castle Rock,NBH_12
4,44,47443.886283,1206.232133,Maine,NBH_36
...,...,...,...,...,...
995,56,NaN,654.708577,Castle Rock,NBH_33
996,50,28142.931306,2009.933452,Castle Rock,NBH_11
997,40,NaN,2646.715248,Maine,NBH_28
998,77,75587.790208,NaN,Castle Rock,NBH_41




### **Exercise:** Build your own synthetic dataset with at least:


> 1. One continuous skewed numeric variable.
> 2. One ordinal categorical variable (e.g., quality rating from 1–5).
> 3. One nominal categorical variable with high cardinality (e.g., 100 IDs).
> 4. Inject at least 10% missingness into one variable, and conditionally add more missingness based on another.



> Write a short description of what each column is supposed to represent in the “world” of your dataset.

Numerical Values

In [ ]:
n = 1000

rng = np.random.default_rng(42)

# Numerical values
mimic_housing = pd.DataFrame(
    data={
        "age": rng.integers(low=5, high=95, size=n),
        "house_size": rng.lognormal(mean=4.5, sigma=0.5, size=n),
        "price": rng.normal(loc=150_000, scale=50_000, size=n),
        "avg_rooms": rng.uniform(low=1.5, high=10.5, size=n),
    }
)
mimic_housing.head()

,age,house_size,price,avg_rooms
0,13,184.307812,178165.653676,6.770434
1,74,94.232025,155729.221851,8.533193
2,63,120.348121,166497.176933,4.895537
3,44,87.497341,188880.476514,1.611575
4,43,82.664993,109525.684055,6.087612


Categorical Values

In [ ]:
cities = ["Castle Rock", "Maine", "BH", "Camboriu", "London"]
mimic_housing["city"] = rng.choice(cities, size=n, p=[0.2, 0.2, 0.15, 0.05, 0.4])

neighbourhoods = [
    "Kobrasol",
    "Barreiros",
    "Tupi",
    "Jatobá",
    "Centro",
    "Floramar",
    "Guarani",
    "Santa Amélia",
]
mimic_housing["neighbourhood"] = rng.choice(neighbourhoods, size=n)

mimic_housing

,age,house_size,price,avg_rooms,city,neighbourhood
0,13,184.307812,178165.653676,6.770434,London,Kobrasol
1,74,94.232025,155729.221851,8.533193,Maine,Kobrasol
2,63,120.348121,166497.176933,4.895537,Camboriu,Kobrasol
3,44,87.497341,188880.476514,1.611575,Castle Rock,Floramar
4,43,82.664993,109525.684055,6.087612,Maine,Jatobá
...,...,...,...,...,...,...
995,61,256.723435,48419.021567,1.938502,London,Kobrasol
996,52,43.442188,160585.689209,1.648421,Maine,Guarani
997,37,74.044125,188107.462098,5.761279,London,Centro
998,91,211.225512,135996.568656,2.979409,BH,Tupi


Missingness

In [ ]:
# 10% missing completely at random
mask = rng.random(n) < 0.1
mimic_housing.loc[mask, "avg_rooms"] = np.nan

mask2 = (mimic_housing["age"] > 90) & (rng.random(n) < 0.2)
mimic_housing.loc[mask2, "house_size"] = np.nan

mimic_housing

,age,house_size,price,avg_rooms,city,neighbourhood
0,13,184.307812,178165.653676,6.770434,London,Kobrasol
1,74,94.232025,155729.221851,8.533193,Maine,Kobrasol
2,63,120.348121,166497.176933,4.895537,Camboriu,Kobrasol
3,44,87.497341,188880.476514,1.611575,Castle Rock,Floramar
4,43,82.664993,109525.684055,6.087612,Maine,Jatobá
...,...,...,...,...,...,...
995,61,256.723435,48419.021567,1.938502,London,Kobrasol
996,52,43.442188,160585.689209,1.648421,Maine,Guarani
997,37,74.044125,188107.462098,5.761279,London,Centro
998,91,NaN,135996.568656,NaN,BH,Tupi


## Pre-Lesson 2B — Strengthening a Synthetic Tabular Dataset

**Objective.** Extend a basic mimicry dataset by (i) introducing an ordinal categorical feature,
(ii) adding a high-cardinality nominal feature, (iii) generating a target that depends on features,
and (iv) injecting missingness with both MCAR and conditional patterns.

**Design notes.**
- **Ordinal** (`quality` 1–5) should correlate with price monotonically.
- **High-card nominal** (`street_id`) trains our encoders to behave under many rare levels.
- **Target generation**: linear-on-transformed features with city/neighbourhood random effects and heteroskedastic noise.
- **Sanity constraints**: nonnegative prices; heavy tails controlled.


In [ ]:
q_probs = [0.05, 0.20, 0.40, 0.25, 0.10]  # Skew toward mid quality
mimic_housing["quality"] = rng.choice([1, 2, 3, 4, 5], size=n, p=q_probs).astype(
    np.int16
)

# High-card nominal: 250 synthetic streets
streets = [f"ST_{i:03d}" for i in range(250)]
mimic_housing["street_id"] = rng.choice(streets, size=n)

mimic_housing

,age,house_size,price,avg_rooms,city,neighbourhood,quality,street_id
0,13,184.307812,178165.653676,6.770434,London,Kobrasol,4,ST_174
1,74,94.232025,155729.221851,8.533193,Maine,Kobrasol,5,ST_193
2,63,120.348121,166497.176933,4.895537,Camboriu,Kobrasol,3,ST_001
3,44,87.497341,188880.476514,1.611575,Castle Rock,Floramar,3,ST_070
4,43,82.664993,109525.684055,6.087612,Maine,Jatobá,4,ST_084
...,...,...,...,...,...,...,...,...
995,61,256.723435,48419.021567,1.938502,London,Kobrasol,2,ST_031
996,52,43.442188,160585.689209,1.648421,Maine,Guarani,2,ST_005
997,37,74.044125,188107.462098,5.761279,London,Centro,4,ST_189
998,91,NaN,135996.568656,NaN,BH,Tupi,4,ST_025


In [ ]:
# city base effects (multiplicative), neighbourhood random noise, plus size/rooms/quality contributions
city_effect = {
    "London": 2.5,
    "Maine": 1,
    "Castle Rock": 1.15,
    "BH": 1.75,
    "Camboriu": 1.15,
}
nbh_effect = {
    nbh: rng.normal(1.0, 0.05) for nbh in mimic_housing["neighbourhood"].unique()
}

# Helper to get effects
ce = mimic_housing["city"].map(city_effect)
ne = mimic_housing["neighbourhood"].map(nbh_effect)

# construct latent log-price; guard against NaNs in house_size/avg_rooms via safe fills for generation
hs = mimic_housing["house_size"].fillna(mimic_housing["house_size"].median())
ar = mimic_housing["avg_rooms"].fillna(mimic_housing["avg_rooms"].median())

# diminishing returns with log(house_size), additive quality, modest rooms effect
log_price_latent = (
    11.0
    + 0.45 * np.log(hs + 1)
    + 0.06 * ar
    + 0.12
    + mimic_housing["quality"]
    + np.log(ce)
    + np.log(ne)
)

eps = rng.normal(
    loc=0.0, scale=0.12 * np.abs(np.log(hs + 1) - np.log(hs + 1).median()), size=n
)
price_generated = np.exp(log_price_latent + eps)

# re-scale to a realistic unit: bring median near 180k
scale = 180_000 / np.median(price_generated)
mimic_housing["price"] = np.maximum(10_000, price_generated * scale)

Inject missingness with both MCAR and conditional patterns

In [ ]:
# 10% MCAR in avg_rooms (kept from your version)
mask_mcar = rng.random(n) < 0.10
mimic_housing.loc[mask_mcar, "avg_rooms"] = np.nan

# additional conditional missingness: very old buildings more likely to miss house_size
mask_cond = (mimic_housing["age"] > 90) & (rng.random(n) < 0.20)
mimic_housing.loc[mask_cond, "house_size"] = np.nan

# optional: MNAR-ish missingness in quality for certain cities
mask_city = (mimic_housing["city"].isin(["London", "BH"])) & (rng.random(n) < 0.05)
mimic_housing.loc[mask_city, "quality"] = np.nan

Quick audit: sanity check

In [15]:
audit = pd.DataFrame(
    {
        "dtype": mimic_housing.dtypes.astype(str),
        "n_unique": mimic_housing.nunique(),
        "n_missing": mimic_housing.isna().sum(),
        "pct_missing": (100 * mimic_housing.isna().mean()).round(1),
    }
).sort_values(["pct_missing", "n_unique"], ascending=[False, True])
audit

,dtype,n_unique,n_missing,pct_missing
avg_rooms,float64,803,197,19.7
quality,float64,5,32,3.2
house_size,float64,986,14,1.4
city,object,5,0,0.0
neighbourhood,object,8,0,0.0
age,int64,90,0,0.0
street_id,object,245,0,0.0
price,float64,998,0,0.0


## Lesson 2 - Data Audit: Profiling, Missingness, and Cardinality

Before modeling, we must **read** the dataset. This stage is often called **data audit** or **profiling**.  
The purpose is not to “fix” anything yet, but to develop a clear map of what’s present, what’s missing, and what looks unusual.  
Think of it as reconnaissance: you are surveying the terrain before you build.

### 2.1 Types of Variables
Every column should be classified properly:
- **Numeric (continuous/discrete).** Examples: square footage, year built, number of rooms.
- **Categorical (nominal/ordinal).** Examples: neighborhood (nominal), quality rating (ordinal).
- **Identifiers.** Columns like IDs, parcel numbers, or addresses are not predictive and should usually be excluded.

Incorrect types are common: integers used as category labels, text fields where ordinals should be, or floats that are really IDs.

### 2.2 Missingness
Missing values are not all the same. Statisticians describe three types:
- **MCAR (Missing Completely at Random).** No pattern; any value could be missing.
- **MAR (Missing at Random).** Missingness depends on other observed variables.
- **MNAR (Missing Not at Random).** Missingness depends on the value itself (e.g., luxury homes less likely to disclose renovations).

Understanding which applies guides how you impute. At the audit stage, you’re simply measuring:
- Total missing values per column.
- Percentage of missingness.
- Whether missingness clusters in certain variables.

### 2.3 Cardinality of Categorical Features
Categorical variables vary in complexity:
- **Low cardinality**: few unique values (e.g., “Yes/No”).
- **Moderate cardinality**: manageable groups (e.g., 10–20 neighborhoods).
- **High cardinality**: many unique values (e.g., thousands of unique zip codes).

High cardinality can be powerful but introduces risks:
- Overfitting (rare levels appear only once).
- Leakage (if a category encodes target information too directly).
- Inefficient encoding (too many one-hot columns).

### 2.4 Low-Variance Traps
Columns that are nearly constant or have extremely skewed distributions provide little signal but can bloat the feature space.  
Part of your audit is flagging them for possible removal or re-encoding.


### 2.5 Audit as Narrative
At the end of the audit, you should be able to write a one-page summary answering:
- What types of variables are present?
- Which columns are most missing, and what risks does this pose?
- Which features are high cardinality and need careful encoding?
- Which features are nearly constant and may be dropped?

Remember: at this stage you are **observing, not altering.**


In [ ]:
from sklearn.datasets import fetch_california_housing

# load data
data = fetch_california_housing(as_frame=True)
california = data.frame.rename(columns={"MedHouseVal": "target"})

# Create some categorical variables for illustration
california["lat_band"] = pd.cut(
    california["Latitude"],
    bins=[-np.inf, 33, 36, 39, np.inf],
    labels=["South", "Central", "North", "FarNorth"],
)

california["is_coastal"] = (california["Longitude"] < -122).map(
    {True: "Coastal", False: "Inland"}
)

# Inject some missingness
rng = np.random.default_rng(42)
mask = rng.random(len(california)) < 0.05
california.loc[mask, "AveRooms"] = np.nan

# Audit summary
audit = pd.DataFrame(
    {
        "dtype": california.dtypes.astype(str),
        "n_unique": california.unique(),
        "n_missing": california.isna().sum(),
        "pct_missing": (100 * california.isna().mean()).round(),
    }
).sort_values(["pct_missing", "n_unique"], ascending=[False, True])

display(audit)

AttributeError: 'DataFrame' object has no attribute 'unique'